In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.metrics import (mean_absolute_percentage_error, mean_absolute_error,
                             root_mean_squared_error, r2_score)

In [2]:
X_train = pd.read_csv("../data/processed/X_train.csv")
y_train_log = pd.read_csv("../data/processed/y_train_log.csv").squeeze("columns")
grup_train = pd.read_csv("../data/processed/grup_train.csv").squeeze("columns")

print(X_train.shape, y_train_log.shape, grup_train.shape)
print("jumlah grup:", grup_train.nunique())

(2533, 111) (2533,) (2533,)
jumlah grup: 2291


In [3]:
gkf = GroupKFold(n_splits=5)

def evaluasi_cv(model, X, y_log, grup):
    hasil = []
    for idx_latih, idx_uji in gkf.split(X, y_log, groups=grup):
        m = clone(model)
        m.fit(X.iloc[idx_latih], y_log.iloc[idx_latih])

        pred = np.expm1(m.predict(X.iloc[idx_uji]))   # log -> rupiah
        asli = np.expm1(y_log.iloc[idx_uji])

        hasil.append({
            "MAPE (%)":    mean_absolute_percentage_error(asli, pred) * 100,
            "MAE (juta)":  mean_absolute_error(asli, pred) / 1e6,
            "RMSE (juta)": root_mean_squared_error(asli, pred) / 1e6,
            "R2":          r2_score(asli, pred),
        })
    return pd.DataFrame(hasil).mean()

In [4]:
MIRING = ["land_size_m2", "building_size_m2", "carports", "garages",
          "maid_bedrooms", "maid_bathrooms", "building_age", "listrik"]

log_fitur = ColumnTransformer(
    [("log", FunctionTransformer(np.log1p), MIRING)],
    remainder="passthrough",
)

hasil = {}
hasil["Baseline: tebak median"] = evaluasi_cv(
    DummyRegressor(strategy="median"), X_train, y_train_log, grup_train)
hasil["Ridge (fitur mentah)"] = evaluasi_cv(
    make_pipeline(StandardScaler(), Ridge(alpha=1.0)), X_train, y_train_log, grup_train)
hasil["Ridge (fitur miring di-log)"] = evaluasi_cv(
    make_pipeline(log_fitur, StandardScaler(), Ridge(alpha=1.0)), X_train, y_train_log, grup_train)

pd.DataFrame(hasil).T.round(2)

,MAPE (%),MAE (juta),RMSE (juta),R2
Baseline: tebak median,91.30,3230.62,10361.29,-0.07
Ridge (fitur mentah),47.21,13323.39,251398.33,-1663.83
Ridge (fitur miring di-log),24.60,1212.76,5339.61,0.74


In [5]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

model_pohon = {
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
    "XGBoost":       XGBRegressor(random_state=42, n_jobs=-1),
    "LightGBM":      LGBMRegressor(random_state=42, verbose=-1),
}

for nama, model in model_pohon.items():
    hasil[nama] = evaluasi_cv(model, X_train, y_train_log, grup_train)

pd.DataFrame(hasil).T.round(2).sort_values("MAPE (%)")

,MAPE (%),MAE (juta),RMSE (juta),R2
XGBoost,24.16,1240.61,5537.09,0.71
LightGBM,24.34,1315.69,5636.07,0.71
Ridge (fitur miring di-log),24.60,1212.76,5339.61,0.74
Random Forest,24.84,1267.51,5471.15,0.72
Ridge (fitur mentah),47.21,13323.39,251398.33,-1663.83
Baseline: tebak median,91.30,3230.62,10361.29,-0.07
